
# India EV Analytics - reproducible analysis notebook

This notebook replaces the saved Colab HTML page of the original project. Every
figure it shows is read from `data/processed/`, which is rebuilt from the raw
workbooks by `python run_all.py`.

**What this notebook demonstrates**

1. The state master: 36 states/UTs with every derived metric.
2. Why the original penetration measure was wrong, and what the corrected
   values are.
3. The statistical analysis: Pearson and Spearman with FDR control, OLS with
   HC3 robust errors, and the sensitivity test that shows the headline income
   correlation is outlier-driven.
4. The forecast: a backtest of four methods and the scenario band that
   replaces the two conflicting point forecasts.


## 0. Setup

In [ ]:

import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

DATA = ROOT / "data" / "processed"
TABLES = ROOT / "outputs" / "tables"
FIGURES = ROOT / "outputs" / "figures"

print("pandas", pd.__version__)



## 1. The validated state master

One row per state/UT. `EV_per_1000_Population` is the corrected penetration
measure: each state's EV count divided by **its own** population.


In [ ]:

master = pd.read_csv(DATA / "state_master.csv")
master.shape


In [ ]:

cols = ["State", "State_Type", "Region", "EV_Registrations_FY21_25", "Total_Registrations_FY21_25",
        "EV_per_1000_Population", "EV_Share_of_Total_Registrations_Pct",
        "Charging_Stations_Total", "Fast_Charger_Share_Pct", "EV_per_Charger",
        "Per_Capita_NSDP", "EV_Readiness_Index", "Readiness_Tier"]
master[cols].head(12)



### 1.1 The denominator bug, demonstrated

The original dashboard computed

    EV / 1,388,166,000 x 1,000     (India's TOTAL population)

for every state, which is why Uttar Pradesh appeared with **0.7235** instead
of **4.26**.  The proof is one line of arithmetic:

    1,004,330 / 1,388,166,000 x 1,000 = 0.7235   <- what the dashboard showed
    1,004,330 /   235,687,000 x 1,000 = 4.2613   <- the correct value


In [ ]:

total_pop = master["Population_2023"].sum()
up = master.loc[master["State"] == "Uttar Pradesh"].iloc[0]

print(f"India's population (the wrong denominator): {total_pop:,}")
print(f"Uttar Pradesh population (the right one) : {up['Population_2023']:,}")
print()
print(f"buggy   : {up['EV_Registrations_FY21_25']:,} / {total_pop:,} * 1000 = "
      f"{up['EV_Registrations_FY21_25'] / total_pop * 1000:.4f}   <- the old dashboard")
print(f"correct : {up['EV_Registrations_FY21_25']:,} / {up['Population_2023']:,} * 1000 = "
      f"{up['EV_per_1000_Population']:.4f}   <- this pipeline")


In [ ]:

# The visual that was published (for the record) versus the corrected one.
states = master.nlargest(12, "EV_Registrations_FY21_25")
buggy = states["EV_Registrations_FY21_25"] / total_pop * 1000
correct = states["EV_per_1000_Population"]

fig, ax = plt.subplots(figsize=(8, 5))
y = range(len(states))
ax.barh([i + 0.2 for i in y], buggy, height=0.38, color="#d1495b", label="Published (wrong denominator)")
ax.barh([i - 0.2 for i in y], correct, height=0.38, color="#2e75b6", label="Corrected")
ax.set_yticks(list(y), states["State"])
ax.set_xlabel("EVs per 1,000 people")
ax.legend()
ax.set_title("The published penetration visual ranked states by raw volume")
plt.tight_layout()
plt.show()



## 2. National picture


In [ ]:

national_ev = master["EV_Registrations_FY21_25"].sum()
national_total = master["Total_Registrations_FY21_25"].sum()
print(f"EV registrations (36 states/UTs) : {national_ev:,}")
print(f"Total registrations               : {national_total:,}")
print(f"EV share of the parc              : {national_ev / national_total * 100:.2f}%")
print(f"EVs per 1,000 people              : {national_ev / total_pop * 1000:.2f}")
print(f"Charging stations                 : {master['Charging_Stations_Total'].sum():,}")
print(f"EVs per charging station          : {national_ev / master['Charging_Stations_Total'].sum():.0f}")


In [ ]:

top = master.nlargest(15, "EV_Registrations_FY21_25").sort_values("EV_Registrations_FY21_25")
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top["State"], top["EV_Registrations_FY21_25"], color="#1f4e79")
ax.set_xlabel("EV registrations, FY2021-22 to FY2025-26")
ax.set_title("Top 15 states by EV volume")
plt.tight_layout()
plt.show()



## 3. Statistics

The original project reported a single Pearson correlation (r = 0.582) between
income and EV penetration with no influence diagnostics.  Here the same number
is reproduced **and** stress-tested.


In [ ]:

corr = pd.read_csv(TABLES / "correlations.csv")
corr[["relationship", "n", "pearson_r", "pearson_p", "pearson_p_fdr",
      "spearman_rho", "spearman_p_fdr", "r_squared"]]


In [ ]:

sens = pd.read_csv(TABLES / "income_penetration_sensitivity.csv")
sens



### 3.1 Why the income effect is not what it looked like

Removing Goa, Chandigarh and Delhi - three small, rich, dense urban
territories - takes the correlation from 0.582 to 0.074.  The relationship is
real but it is *not* a general income effect; it is an urban-territory effect.

The pipeline runs the same test automatically and records it as a
**critical** data-quality warning, so it cannot be quietly dropped.


In [ ]:

d = master.dropna(subset=["Per_Capita_NSDP"])
outliers = ["Goa", "Chandigarh", "Delhi"]

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.scatter(d.loc[~d.State.isin(outliers), "Per_Capita_NSDP"],
           d.loc[~d.State.isin(outliers), "EV_per_1000_Population"],
           color="#2e75b6", s=52, edgecolor="white", label="States")
ax.scatter(d.loc[d.State.isin(outliers), "Per_Capita_NSDP"],
           d.loc[d.State.isin(outliers), "EV_per_1000_Population"],
           color="#d1495b", s=95, edgecolor="white", zorder=5, label="High-leverage outliers")
for _, r in d[d.State.isin(outliers)].iterrows():
    ax.annotate(r["State"], (r["Per_Capita_NSDP"], r["EV_per_1000_Population"]),
                textcoords="offset points", xytext=(9, 4), fontsize=8, color="#d1495b")
ax.set_xlabel("Per-capita NSDP (INR, 2023-24)")
ax.set_ylabel("EVs per 1,000 people")
ax.legend()
plt.tight_layout()
plt.show()



### 3.2 Regression models with robust errors


In [ ]:

models = pd.read_csv(TABLES / "model_summary.csv")
models[["model", "n", "r_squared", "adj_r_squared", "f_pvalue", "rmse", "max_cooks_d"]]


In [ ]:

coefs = pd.read_csv(TABLES / "model_coefficients.csv")
m2 = coefs[coefs.model.str.startswith("M2 ")]
m2[["term", "coef", "std_error", "t", "p_value", "ci_low", "ci_high", "std_beta"]]



Model M2 adds the infrastructure variables the original project owned but never
modelled.  R-squared rises from 0.339 (income alone) to
`R^2` and the income coefficient loses significance - the honest reading
is that *infrastructure provision explains adoption better than income does*.



## 4. Forecast

Four methods are backtested on the last **complete** financial year.  The
best of them still misses by 40% MAPE, which is why the project now publishes
a scenario band instead of a single number.


In [ ]:

bt = pd.read_csv(TABLES / "forecast_backtest.csv")
bt


In [ ]:

sc = pd.read_csv(TABLES / "forecast_scenarios.csv")
piv = sc.pivot(index="Financial_Year", columns="Scenario", values="EV_Registrations")
piv[["Low", "Base", "High"]]


In [ ]:

annual = pd.read_csv(TABLES / "forecast_annual.csv")
piv2 = sc.pivot(index="Financial_Year", columns="Scenario", values="EV_Registrations")
n_hist = int(annual["Is_Complete_Year"].fillna(False).sum())

fig, ax = plt.subplots(figsize=(9, 4.8))
x = np.arange(len(piv2))
ax.fill_between(x, piv2["Low"], piv2["High"], color="#e8871a", alpha=0.18)
ax.plot(x, piv2["Base"], color="#e8871a", linewidth=2.4, marker="o", label="Base (15% CAGR)")
ax.plot(x, piv2["Low"], color="#7f8c8d", linestyle="--", linewidth=1.2, label="Low (5%)")
ax.plot(x, piv2["High"], color="#1f4e79", linestyle="--", linewidth=1.2, label="High (25%)")
ax.plot(np.arange(n_hist), annual["Actual"].iloc[:n_hist], color="#1f4e79",
        linewidth=2.6, marker="o", label="Actual (complete years)")
ax.scatter([n_hist], [annual["Actual"].iloc[n_hist]], color="#d1495b", s=110,
           marker="X", zorder=6, label="FY2025-26 reported (part year)")
ax.set_xticks(x, piv2.index, rotation=30)
ax.legend(fontsize=8)
ax.set_title("Actual EV registrations and the scenario fan")
plt.tight_layout()
plt.show()


In [ ]:

recon = pd.read_csv(TABLES / "forecast_reconciliation.csv")
recon



## 5. Reproducing this notebook

```bash
python run_all.py          # rebuild every artefact from the raw workbooks
python -m pytest tests -q  # 26 regression tests
```

The full technical report is at `outputs/report/EV_India_Analytics_Report.pdf`
and the interactive dashboard at `outputs/dashboard/ev_india_dashboard.html`.
